# 3. Post-Submission Review

从提交结果 (`data/my_trades/`) 中加载 orderbook 快照、我方成交、PnL，绘制复盘图：

1. **主图**：Orderbook 散点 + 市场成交 + 我方买卖（共享横轴）
2. **仓位图**：逐笔累计仓位变化
3. **PnL 曲线**：总 PnL + 分产品 PnL
4. **交易质量分析**：成交偏离 fair 的分布、fill rate、每笔 edge 统计

In [31]:
import sys, json, io
sys.path.insert(0, '.')

from pathlib import Path
from utils.orderbook import compute_wall_mid
import polars as pl
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Step 1: 加载提交数据

从 `.log` 文件解析：`activitiesLog`（orderbook 快照）、`tradeHistory`（所有成交）、从 `.json` 解析 `graphLog`（总 PnL 曲线）。

In [32]:
# === 配置 ===
SUBMISSION_ID = "165927"
MY_TRADES_DIR = Path("data/my_trades")

# 加载 .log（含 tradeHistory）和 .json（含 graphLog / profit）
with open(MY_TRADES_DIR / f"{SUBMISSION_ID}.log") as f:
    log_data = json.loads(f.read())
with open(MY_TRADES_DIR / f"{SUBMISSION_ID}.json") as f:
    json_data = json.load(f)

print(f"Round: {json_data['round']}, Status: {json_data['status']}, Final Profit: {json_data['profit']}")
print(f"Final positions: {json_data['positions']}")

Round: 1, Status: FINISHED, Final Profit: 7563.0
Final positions: [{'symbol': 'INTARIAN_PEPPER_ROOT', 'quantity': 80}, {'symbol': 'XIRECS', 'quantity': -960429}]


## Step 2: 解析数据

In [33]:
# --- 1) Orderbook 快照（宽格式 + 长格式）---
ob_wide = pl.read_csv(io.StringIO(log_data["activitiesLog"]), separator=";")

# 将价格/数量列统一转成数值，兼容日志里字符串类型
numeric_cols = [
    c for c in ob_wide.columns
    if c in {"timestamp", "day", "mid_price", "profit_and_loss"}
    or c.startswith("bid_price_")
    or c.startswith("ask_price_")
    or c.startswith("bid_volume_")
    or c.startswith("ask_volume_")
]
if numeric_cols:
    ob_wide = ob_wide.with_columns([pl.col(c).cast(pl.Float64, strict=False).alias(c) for c in numeric_cols])
    if "timestamp" in ob_wide.columns:
        ob_wide = ob_wide.with_columns(pl.col("timestamp").cast(pl.Int64, strict=False).alias("timestamp"))

# 宽格式 -> 长格式（每档一行），用于散点图
rows = []
for side in ["bid", "ask"]:
    for level in range(1, 4):
        pc, vc = f"{side}_price_{level}", f"{side}_volume_{level}"
        if pc in ob_wide.columns:
            rows.append(
                ob_wide.select([
                    "day", "timestamp", "product",
                    pl.col(pc).cast(pl.Float64, strict=False).alias("price"),
                    pl.col(vc).cast(pl.Float64, strict=False).alias("volume"),
                    pl.lit(side).alias("side"),
                    pl.lit(level).alias("level"),
                    "mid_price", "profit_and_loss",
                ])
            )

if rows:
    ob_long = (
        pl.concat(rows, how="vertical_relaxed")
        .filter(pl.col("price").is_not_null())
        .sort(["timestamp", "product", "side", "level"])
    )
else:
    ob_long = pl.DataFrame(
        schema={
            "day": pl.Float64,
            "timestamp": pl.Int64,
            "product": pl.String,
            "price": pl.Float64,
            "volume": pl.Float64,
            "side": pl.String,
            "level": pl.Int64,
            "mid_price": pl.Float64,
            "profit_and_loss": pl.Float64,
        }
    )

# --- 2) 成交记录 ---
trade_history = log_data.get("tradeHistory", [])
if len(trade_history) == 0:
    all_trades = pl.DataFrame(
        schema={
            "product": pl.String,
            "price": pl.Float64,
            "quantity": pl.Int64,
            "buyer": pl.String,
            "seller": pl.String,
            "timestamp": pl.Int64,
        }
    )
else:
    all_trades = pl.DataFrame(trade_history)
    if "symbol" in all_trades.columns:
        all_trades = all_trades.rename({"symbol": "product"})
    for col_name, dtype in [
        ("product", pl.String),
        ("price", pl.Float64),
        ("quantity", pl.Int64),
        ("buyer", pl.String),
        ("seller", pl.String),
        ("timestamp", pl.Int64),
    ]:
        if col_name not in all_trades.columns:
            all_trades = all_trades.with_columns(pl.lit(None, dtype=dtype).alias(col_name))

# 分类：我方 vs 市场
all_trades = all_trades.with_columns([
    pl.when(pl.col("buyer") == "SUBMISSION").then(pl.lit("my_buy"))
      .when(pl.col("seller") == "SUBMISSION").then(pl.lit("my_sell"))
      .otherwise(pl.lit("market"))
      .alias("trade_type"),
])

my_trades = all_trades.filter(pl.col("trade_type") != "market")
mkt_trades = all_trades.filter(pl.col("trade_type") == "market")

# --- 3) PnL 曲线（总额）---
pnl_total = pl.read_csv(io.StringIO(json_data["graphLog"]), separator=";")

# --- 4) Fair 计算 ---
products = ob_wide["product"].unique().to_list()
wall_mid_df = compute_wall_mid(ob_wide)

# 默认 fair 用 wall_mid
fair_df = wall_mid_df.rename({"wall_mid": "fair"})


def _pick_max_vol_price(row: dict, side: str, vol_threshold: float = 20.0):
    """选择该侧订单量绝对值最大且大于阈值的报价。"""
    candidates = []
    for level in range(1, 4):
        p = row.get(f"{side}_price_{level}")
        v = row.get(f"{side}_volume_{level}")
        if p is None or v is None:
            continue
        v_abs = abs(float(v))
        if v_abs > vol_threshold:
            candidates.append((v_abs, float(p)))

    if not candidates:
        return None

    max_vol = max(v for v, _ in candidates)
    top_prices = [p for v, p in candidates if v == max_vol]
    if side == "bid":
        return max(top_prices)
    return min(top_prices)


# ASH_COATED_OSMIUM: fair = (bid_max_vol_gt20 + ask_max_vol_gt20)/2
# 若某一侧当下缺失，则使用过去最近时刻该侧价格。
ash = (
    ob_wide.filter(pl.col("product") == "ASH_COATED_OSMIUM")
    .sort("timestamp")
    .select([
        "timestamp", "product", "mid_price",
        "bid_price_1", "bid_volume_1", "bid_price_2", "bid_volume_2", "bid_price_3", "bid_volume_3",
        "ask_price_1", "ask_volume_1", "ask_price_2", "ask_volume_2", "ask_price_3", "ask_volume_3",
    ])
)

ash_rows = []
last_bid = None
last_ask = None
for row in ash.iter_rows(named=True):
    bid_px = _pick_max_vol_price(row, "bid", vol_threshold=20.0)
    ask_px = _pick_max_vol_price(row, "ask", vol_threshold=20.0)

    use_bid = bid_px if bid_px is not None else last_bid
    use_ask = ask_px if ask_px is not None else last_ask

    if bid_px is not None:
        last_bid = bid_px
    if ask_px is not None:
        last_ask = ask_px

    if use_bid is not None and use_ask is not None:
        ash_fair = (use_bid + use_ask) / 2.0
    else:
        # 在最早阶段尚无“过去最近时刻”可用时，退回 mid_price 兜底
        ash_fair = float(row["mid_price"]) if row.get("mid_price") is not None else None

    ash_rows.append({
        "timestamp": row["timestamp"],
        "product": "ASH_COATED_OSMIUM",
        "fair_ash": ash_fair,
    })

ash_fair_df = pl.DataFrame(ash_rows)

# 辣椒根 fair: int(12000 + 0.001 * t)
fair_df = (
    fair_df.join(ash_fair_df, on=["timestamp", "product"], how="left")
    .with_columns(
        pl.when(pl.col("product") == "INTARIAN_PEPPER_ROOT")
          .then((pl.lit(12000.0) + pl.col("timestamp") * pl.lit(0.001)).floor().cast(pl.Int64).cast(pl.Float64))
          .when(pl.col("product") == "ASH_COATED_OSMIUM")
          .then(pl.coalesce([pl.col("fair_ash"), pl.col("fair")]))
          .otherwise(pl.col("fair"))
          .alias("fair")
    )
    .drop("fair_ash")
)

print(f"Orderbook snapshots: {ob_wide.height}, Products: {products}")
print(f"My trades: {my_trades.height} ({my_trades.filter(pl.col('trade_type')=='my_buy').height} buys, "
      f"{my_trades.filter(pl.col('trade_type')=='my_sell').height} sells)")
print(f"Market trades: {mkt_trades.height}")
print(f"PnL datapoints: {pnl_total.height}")
my_trades.head(5)

Orderbook snapshots: 2000, Products: ['ASH_COATED_OSMIUM', 'INTARIAN_PEPPER_ROOT']
My trades: 24 (16 buys, 8 sells)
Market trades: 82
PnL datapoints: 500


timestamp,buyer,seller,product,currency,price,quantity,trade_type
i64,str,str,str,str,f64,i64,str
0,"""SUBMISSION""","""""","""INTARIAN_PEPPER_ROOT""","""XIRECS""",12006.0,11,"""my_buy"""
200,"""SUBMISSION""","""""","""INTARIAN_PEPPER_ROOT""","""XIRECS""",12007.0,10,"""my_buy"""
300,"""SUBMISSION""","""""","""INTARIAN_PEPPER_ROOT""","""XIRECS""",12007.0,11,"""my_buy"""
400,"""SUBMISSION""","""""","""INTARIAN_PEPPER_ROOT""","""XIRECS""",12007.0,9,"""my_buy"""
500,"""SUBMISSION""","""""","""INTARIAN_PEPPER_ROOT""","""XIRECS""",12007.0,10,"""my_buy"""


In [34]:
# --- 5) 计算逐笔仓位序列 ---
# 我方买入 = +quantity，我方卖出 = -quantity
position_rows = []
for product in products:
    pt = my_trades.filter(pl.col("product") == product).sort("timestamp")
    cum_pos = 0
    # 初始仓位 = 0
    position_rows.append({"timestamp": 0, "product": product, "position": 0})
    for row in pt.iter_rows(named=True):
        signed_qty = row["quantity"] if row["trade_type"] == "my_buy" else -row["quantity"]
        cum_pos += signed_qty
        position_rows.append({"timestamp": row["timestamp"], "product": product, "position": cum_pos})

position_df = pl.DataFrame(position_rows).sort(["product", "timestamp"])

# --- 6) 分产品 PnL（从 activitiesLog 的 profit_and_loss 列）---
pnl_by_product = (
    ob_wide.select(["timestamp", "product", "profit_and_loss"])
    .sort(["product", "timestamp"])
)

# --- 7) 每笔我方成交加上 fair / edge（后续 normalized 图 + Step 5 edge 分析都会用到）---
my_with_wm = my_trades.join(
    fair_df.select(["timestamp", "product", "fair"]),
    on=["timestamp", "product"],
    how="left",
).with_columns(
    pl.when(pl.col("trade_type") == "my_buy")
      .then(pl.col("fair") - pl.col("price"))
      .otherwise(pl.col("price") - pl.col("fair"))
      .alias("edge"),
).with_columns(
    (pl.col("edge") * pl.col("quantity")).alias("edge_pnl"),
)


def print_trade_stats(df: pl.DataFrame, product: str) -> None:
    """Safe printer for trade stats when there are zero fills."""
    print(f"\n{'='*40} {product} {'='*40}")
    print(f"  Total trades: {df.height}")

    if df.height == 0:
        print("  Positive edge trades: 0")
        print("  Zero edge trades: 0")
        print("  Negative edge trades: 0")
        print("  Avg edge: N/A (no fills)")
        print("  Total edge PnL (sum of edge*qty): 0.00")
        print("  Edge distribution: empty")
        return

    pos_cnt = df.filter(pl.col("edge") > 0).height
    zero_cnt = df.filter(pl.col("edge") == 0).height
    neg_cnt = df.filter(pl.col("edge") < 0).height
    avg_edge = float(df["edge"].mean()) if df["edge"].mean() is not None else 0.0
    edge_pnl_sum = float(df["edge_pnl"].sum()) if df["edge_pnl"].sum() is not None else 0.0

    print(f"  Positive edge trades: {pos_cnt}")
    print(f"  Zero edge trades: {zero_cnt}")
    print(f"  Negative edge trades: {neg_cnt}")
    print(f"  Avg edge: {avg_edge:.2f}")
    print(f"  Total edge PnL (sum of edge*qty): {edge_pnl_sum:.2f}")
    print("  Edge distribution:")
    edge_dist = df.group_by("edge").agg([
        pl.len().alias("count"),
        pl.col("quantity").sum().alias("total_qty"),
    ]).sort("edge")
    print(edge_dist)


print("Position range per product:")
for product in products:
    pp = position_df.filter(pl.col("product") == product)
    print(f"  {product}: [{pp['position'].min()}, {pp['position'].max()}]")

print("\nFinal PnL per product:")
for product in products:
    pp = pnl_by_product.filter(pl.col("product") == product).sort("timestamp")
    if pp.height == 0:
        print(f"  {product}: N/A")
    else:
        print(f"  {product}: {pp['profit_and_loss'].to_list()[-1]:.2f}")

Position range per product:
  ASH_COATED_OSMIUM: [0, 0]
  INTARIAN_PEPPER_ROOT: [0, 80]

Final PnL per product:
  ASH_COATED_OSMIUM: 0.00
  INTARIAN_PEPPER_ROOT: 7563.00


## Step 3: 复盘主图

每个产品绘制三行子图（共享横轴）：
- **Row 1**：Orderbook 散点 + Wall Mid + 市场成交 + 我方买卖
- **Row 2**：仓位变化（阶梯线）
- **Row 3**：分产品 PnL 曲线

In [35]:
def plot_review(product: str) -> go.Figure:
    """三行子图：Orderbook+Trades / Position / PnL，共享横轴。"""

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        row_heights=[0.55, 0.2, 0.25],
        subplot_titles=[
            f"{product} — Orderbook + Trades",
            "Position",
            "PnL",
        ],
    )

    # ===== Row 1: Orderbook 散点 =====
    ob_p = ob_long.filter(pl.col("product") == product)
    max_vol = ob_p["volume"].max() or 1

    bids = ob_p.filter(pl.col("side") == "bid")
    asks = ob_p.filter(pl.col("side") == "ask")

    fig.add_trace(go.Scatter(
        x=bids["timestamp"].to_list(), y=bids["price"].to_list(),
        mode="markers",
        marker=dict(color="steelblue", size=[max(2, v / max_vol * 14) for v in bids["volume"].to_list()], opacity=0.4),
        name="Bids", legendgroup="ob",
        hovertemplate="t=%{x}<br>p=%{y}<extra>Bid</extra>",
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=asks["timestamp"].to_list(), y=asks["price"].to_list(),
        mode="markers",
        marker=dict(color="salmon", size=[max(2, v / max_vol * 14) for v in asks["volume"].to_list()], opacity=0.4),
        name="Asks", legendgroup="ob",
        hovertemplate="t=%{x}<br>p=%{y}<extra>Ask</extra>",
    ), row=1, col=1)

    # Fair line
    fair_p = fair_df.filter(pl.col("product") == product).sort("timestamp")
    fig.add_trace(go.Scatter(
        x=fair_p["timestamp"].to_list(), y=fair_p["fair"].to_list(),
        mode="lines", line=dict(color="green", width=1.5),
        name="Fair", legendgroup="fair",
    ), row=1, col=1)

    # 市场成交（黑色 x）
    mt = mkt_trades.filter(pl.col("product") == product)
    if mt.height > 0:
        fig.add_trace(go.Scatter(
            x=mt["timestamp"].to_list(), y=mt["price"].to_list(),
            mode="markers", marker=dict(color="black", symbol="x", size=7),
            name="Market Trades", legendgroup="mkt",
            customdata=mt["quantity"].to_list(),
            hovertemplate="t=%{x}<br>p=%{y}<br>qty=%{customdata}<extra>Mkt</extra>",
        ), row=1, col=1)

    # 我方买入（绿色三角上）
    my_buys = my_trades.filter((pl.col("product") == product) & (pl.col("trade_type") == "my_buy"))
    if my_buys.height > 0:
        fig.add_trace(go.Scatter(
            x=my_buys["timestamp"].to_list(), y=my_buys["price"].to_list(),
            mode="markers",
            marker=dict(color="limegreen", symbol="triangle-up", size=10, line=dict(width=1, color="darkgreen")),
            name="My Buy", legendgroup="my",
            customdata=my_buys["quantity"].to_list(),
            hovertemplate="t=%{x}<br>p=%{y}<br>qty=%{customdata}<extra>My Buy</extra>",
        ), row=1, col=1)

    # 我方卖出（红色三角下）
    my_sells = my_trades.filter((pl.col("product") == product) & (pl.col("trade_type") == "my_sell"))
    if my_sells.height > 0:
        fig.add_trace(go.Scatter(
            x=my_sells["timestamp"].to_list(), y=my_sells["price"].to_list(),
            mode="markers",
            marker=dict(color="red", symbol="triangle-down", size=10, line=dict(width=1, color="darkred")),
            name="My Sell", legendgroup="my",
            customdata=my_sells["quantity"].to_list(),
            hovertemplate="t=%{x}<br>p=%{y}<br>qty=%{customdata}<extra>My Sell</extra>",
        ), row=1, col=1)

    # ===== Row 2: 仓位 =====
    pos_p = position_df.filter(pl.col("product") == product).sort("timestamp")
    fig.add_trace(go.Scatter(
        x=pos_p["timestamp"].to_list(), y=pos_p["position"].to_list(),
        mode="lines", line=dict(color="purple", width=1.5, shape="hv"),
        name="Position", legendgroup="pos",
        hovertemplate="t=%{x}<br>pos=%{y}<extra></extra>",
    ), row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=0.5, row=2, col=1)

    # ===== Row 3: PnL =====
    pnl_p = pnl_by_product.filter(pl.col("product") == product).sort("timestamp")
    fig.add_trace(go.Scatter(
        x=pnl_p["timestamp"].to_list(), y=pnl_p["profit_and_loss"].to_list(),
        mode="lines", line=dict(color="orange", width=1.5),
        name=f"PnL ({product})", legendgroup="pnl",
        hovertemplate="t=%{x}<br>PnL=%{y:.1f}<extra></extra>",
    ), row=3, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=0.5, row=3, col=1)

    # Layout
    fig.update_layout(
        height=900, width=1200,
        legend=dict(orientation="h", y=1.02, x=0, font=dict(size=10)),
        hovermode="x unified",
    )
    fig.update_xaxes(title_text="Timestamp", row=3, col=1)
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Pos", row=2, col=1)
    fig.update_yaxes(title_text="PnL", row=3, col=1)

    return fig

In [36]:
fig = plot_review("INTARIAN_PEPPER_ROOT")
fig.show()

In [37]:
fig = plot_review("ASH_COATED_OSMIUM")
fig.show()

### TOMATOES — Normalized View (去 wall_mid 漂移)

把所有价格减去成交时刻的 wall_mid，去除漂移后观察：
- 挂单 / 成交是否稳定落在 `± 7, ± 6, ± 1~2, 0` 等标准档位
- 我方成交相对 fair 的分布：buy 应该在 ≤ 0 的位置，sell 应该在 ≥ 0 的位置
- 负 edge 的成交会立刻暴露（buy 在 > 0 或 sell 在 < 0）

参考 [utils/viz.py](utils/viz.py) 中的 `plot_normalized_orderbook`，保留所有元素（bids / asks / market trades / my buys / my sells）+ 叠加仓位 / PnL 两行子图。

In [38]:
def plot_review_normalized(product: str) -> go.Figure:
    """和 plot_review 一样的三行布局，但 Row 1 使用 price - fair（去基准）。"""

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        row_heights=[0.7, 0.2, 0.25],
        subplot_titles=[
            f"{product} — Normalized Orderbook (price - fair)",
            "Position",
            "PnL",
        ],
    )

    # ===== Row 1: 去基准 Orderbook =====
    fair_ref = fair_df.filter(pl.col("product") == product).select(["timestamp", "product", "fair"] )

    # 挂单 - fair
    ob_p = (
        ob_long.filter(pl.col("product") == product)
        .join(fair_ref, on=["timestamp", "product"], how="left")
        .filter(pl.col("fair").is_not_null())
        .with_columns((pl.col("price") - pl.col("fair")).alias("norm_price"))
    )
    max_vol = ob_p["volume"].max() or 1

    bids = ob_p.filter(pl.col("side") == "bid")
    fig.add_trace(go.Scatter(
        x=bids["timestamp"].to_list(), y=bids["norm_price"].to_list(),
        mode="markers",
        marker=dict(color="steelblue", size=[max(2, v / max_vol * 14) for v in bids["volume"].to_list()], opacity=0.4),
        name="Bids", legendgroup="ob",
        customdata=bids["volume"].to_list(),
        hovertemplate="t=%{x}<br>Δ=%{y}<br>v=%{customdata}<extra>Bid</extra>",
    ), row=1, col=1)

    asks = ob_p.filter(pl.col("side") == "ask")
    fig.add_trace(go.Scatter(
        x=asks["timestamp"].to_list(), y=asks["norm_price"].to_list(),
        mode="markers",
        marker=dict(color="salmon", size=[max(2, v / max_vol * 14) for v in asks["volume"].to_list()], opacity=0.4),
        name="Asks", legendgroup="ob",
        customdata=asks["volume"].to_list(),
        hovertemplate="t=%{x}<br>Δ=%{y}<br>v=%{customdata}<extra>Ask</extra>",
    ), row=1, col=1)

    # 标记跨越 fair 的非理性限价挂单，并标注订单大小
    irrational_orders = ob_p.filter(
        ((pl.col("side") == "bid") & (pl.col("norm_price") > 0))
        | ((pl.col("side") == "ask") & (pl.col("norm_price") < 0))
    )
    if irrational_orders.height > 0:
        irr_text = [
            f"qty={abs(v):.0f}" if v is not None else "qty=NA"
            for v in irrational_orders["volume"].to_list()
        ]
        irr_sizes = [
            max(10, min(28, abs(v) / max_vol * 30)) if v is not None else 10
            for v in irrational_orders["volume"].to_list()
        ]
        fig.add_trace(go.Scatter(
            x=irrational_orders["timestamp"].to_list(),
            y=irrational_orders["norm_price"].to_list(),
            mode="markers+text",
            marker=dict(
                color="gold",
                symbol="x",
                size=irr_sizes,
                line=dict(width=1, color="black"),
            ),
            text=irr_text,
            textposition="top center",
            textfont=dict(size=9, color="black"),
            name="Cross-Fair Orders",
            legendgroup="irr",
            hovertemplate="t=%{x}<br>Δ=%{y}<br>%{text}<extra>Cross-Fair</extra>",
        ), row=1, col=1)

    # Fair 基准线 (y=0)
    fig.add_hline(y=0, line_color="green", line_dash="dash", line_width=1, row=1, col=1,
                  annotation_text="Fair")

    # 市场成交
    mt = (
        mkt_trades.filter(pl.col("product") == product)
        .join(fair_ref, on=["timestamp", "product"], how="left")
        .filter(pl.col("fair").is_not_null())
        .with_columns((pl.col("price") - pl.col("fair")).alias("norm_price"))
    )
    if mt.height > 0:
        fig.add_trace(go.Scatter(
            x=mt["timestamp"].to_list(), y=mt["norm_price"].to_list(),
            mode="markers", marker=dict(color="black", symbol="x", size=7),
            name="Market Trades", legendgroup="mkt",
            customdata=mt["quantity"].to_list(),
            hovertemplate="t=%{x}<br>Δ=%{y}<br>qty=%{customdata}<extra>Mkt</extra>",
        ), row=1, col=1)

    # 我方买入 / 卖出（用 my_with_wm 里已算好的 fair 作为参照）
    my_p = my_with_wm.filter(pl.col("product") == product).with_columns(
        (pl.col("price") - pl.col("fair")).alias("norm_price")
    )

    my_buys = my_p.filter(pl.col("trade_type") == "my_buy")
    if my_buys.height > 0:
        fig.add_trace(go.Scatter(
            x=my_buys["timestamp"].to_list(), y=my_buys["norm_price"].to_list(),
            mode="markers",
            marker=dict(color="limegreen", symbol="triangle-up", size=10, line=dict(width=1, color="darkgreen")),
            name="My Buy", legendgroup="my",
            customdata=list(zip(my_buys["quantity"].to_list(), my_buys["price"].to_list(), my_buys["edge"].to_list())),
            hovertemplate="t=%{x}<br>Δ=%{y}<br>qty=%{customdata[0]}<br>px=%{customdata[1]}<br>edge=%{customdata[2]}<extra>My Buy</extra>",
        ), row=1, col=1)

    my_sells = my_p.filter(pl.col("trade_type") == "my_sell")
    if my_sells.height > 0:
        fig.add_trace(go.Scatter(
            x=my_sells["timestamp"].to_list(), y=my_sells["norm_price"].to_list(),
            mode="markers",
            marker=dict(color="red", symbol="triangle-down", size=10, line=dict(width=1, color="darkred")),
            name="My Sell", legendgroup="my",
            customdata=list(zip(my_sells["quantity"].to_list(), my_sells["price"].to_list(), my_sells["edge"].to_list())),
            hovertemplate="t=%{x}<br>Δ=%{y}<br>qty=%{customdata[0]}<br>px=%{customdata[1]}<br>edge=%{customdata[2]}<extra>My Sell</extra>",
        ), row=1, col=1)

    # 右侧副轴：fair 趋势
    fair_sorted = fair_ref.sort("timestamp")
    fig.add_trace(go.Scatter(
        x=fair_sorted["timestamp"].to_list(), y=fair_sorted["fair"].to_list(),
        mode="lines", line=dict(color="green", width=1.2, dash="dot"),
        name="Fair Trend", legendgroup="fair",
        yaxis="y4",
        hovertemplate="t=%{x}<br>fair=%{y}<extra></extra>",
    ))

    # ===== Row 2: 仓位 =====
    pos_p = position_df.filter(pl.col("product") == product).sort("timestamp")
    fig.add_trace(go.Scatter(
        x=pos_p["timestamp"].to_list(), y=pos_p["position"].to_list(),
        mode="lines", line=dict(color="purple", width=1.5, shape="hv"),
        name="Position", legendgroup="pos",
        hovertemplate="t=%{x}<br>pos=%{y}<extra></extra>",
    ), row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=0.5, row=2, col=1)

    # ===== Row 3: PnL =====
    pnl_p = pnl_by_product.filter(pl.col("product") == product).sort("timestamp")
    fig.add_trace(go.Scatter(
        x=pnl_p["timestamp"].to_list(), y=pnl_p["profit_and_loss"].to_list(),
        mode="lines", line=dict(color="orange", width=1.5),
        name=f"PnL ({product})", legendgroup="pnl",
        hovertemplate="t=%{x}<br>PnL=%{y:.1f}<extra></extra>",
    ), row=3, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=0.5, row=3, col=1)

    # 配置副轴 (fair trend)
    fig.update_layout(
        yaxis4=dict(
            overlaying="y", side="right", title="Fair",
            showgrid=False, anchor="x",
        ),
        height=900, width=1200,
        legend=dict(orientation="h", y=1.02, x=0, font=dict(size=10)),
        hovermode="x unified",
    )
    fig.update_xaxes(title_text="Timestamp", row=3, col=1)
    fig.update_yaxes(title_text="Δ from Fair (ticks)", row=1, col=1)
    fig.update_yaxes(title_text="Pos", row=2, col=1)
    fig.update_yaxes(title_text="PnL", row=3, col=1)

    return fig


fig = plot_review_normalized("INTARIAN_PEPPER_ROOT")
fig.show()

## Step 4: 总 PnL 曲线

graphLog 提供的全产品汇总 PnL 曲线，叠加分产品 PnL 对比。

In [39]:
# my_with_wm 已在 Step 2 里算好（含 fair / edge / edge_pnl），这里直接做统计
for product in products:
    pt = my_with_wm.filter(pl.col("product") == product)
    print_trade_stats(pt, product)


======================================== ASH_COATED_OSMIUM ========================================
  Total trades: 0
  Positive edge trades: 0
  Zero edge trades: 0
  Negative edge trades: 0
  Avg edge: N/A (no fills)
  Total edge PnL (sum of edge*qty): 0.00
  Edge distribution: empty

======================================== INTARIAN_PEPPER_ROOT ========================================
  Total trades: 24
  Positive edge trades: 16
  Zero edge trades: 0
  Negative edge trades: 8
  Avg edge: 1.29
  Total edge PnL (sum of edge*qty): -312.00
  Edge distribution:
shape: (6, 3)
┌──────┬───────┬───────────┐
│ edge ┆ count ┆ total_qty │
│ ---  ┆ ---   ┆ ---       │
│ f64  ┆ u32   ┆ i64       │
╞══════╪═══════╪═══════════╡
│ -7.0 ┆ 7     ┆ 69        │
│ -6.0 ┆ 1     ┆ 11        │
│ 4.0  ┆ 5     ┆ 15        │
│ 5.0  ┆ 3     ┆ 6         │
│ 6.0  ┆ 7     ┆ 14        │
│ 9.0  ┆ 1     ┆ 7         │
└──────┴───────┴───────────┘


## Step 5: 交易质量分析

### 5a — 每笔成交的 Edge（trade_price - fair）

正 edge = 赚钱方向成交。对 EMERALDS fair=10000 固定，对 TOMATOES 用成交时刻的 wall_mid。

In [40]:
# 给每笔我方成交加上 fair，计算 edge
my_with_wm = my_trades.join(
    fair_df.select(["timestamp", "product", "fair"]),
    on=["timestamp", "product"],
    how="left",
)

# edge = 买入 → fair - price（买越低 edge 越正）；卖出 → price - fair
my_with_wm = my_with_wm.with_columns(
    pl.when(pl.col("trade_type") == "my_buy")
      .then(pl.col("fair") - pl.col("price"))
      .otherwise(pl.col("price") - pl.col("fair"))
      .alias("edge"),
).with_columns(
    (pl.col("edge") * pl.col("quantity")).alias("edge_pnl"),  # edge × qty
)

for product in products:
    pt = my_with_wm.filter(pl.col("product") == product)
    print_trade_stats(pt, product)


======================================== ASH_COATED_OSMIUM ========================================
  Total trades: 0
  Positive edge trades: 0
  Zero edge trades: 0
  Negative edge trades: 0
  Avg edge: N/A (no fills)
  Total edge PnL (sum of edge*qty): 0.00
  Edge distribution: empty

======================================== INTARIAN_PEPPER_ROOT ========================================
  Total trades: 24
  Positive edge trades: 16
  Zero edge trades: 0
  Negative edge trades: 8
  Avg edge: 1.29
  Total edge PnL (sum of edge*qty): -312.00
  Edge distribution:
shape: (6, 3)
┌──────┬───────┬───────────┐
│ edge ┆ count ┆ total_qty │
│ ---  ┆ ---   ┆ ---       │
│ f64  ┆ u32   ┆ i64       │
╞══════╪═══════╪═══════════╡
│ -7.0 ┆ 7     ┆ 69        │
│ -6.0 ┆ 1     ┆ 11        │
│ 4.0  ┆ 5     ┆ 15        │
│ 5.0  ┆ 3     ┆ 6         │
│ 6.0  ┆ 7     ┆ 14        │
│ 9.0  ┆ 1     ┆ 7         │
└──────┴───────┴───────────┘


### 5b — Edge 散点时序图

每笔成交的 edge 随时间变化，观察是否有时段 edge 恶化。

In [41]:
fig = make_subplots(rows=len(products), cols=1, shared_xaxes=True,
                    subplot_titles=[f"{p} — Edge per Trade" for p in products],
                    vertical_spacing=0.08)

for i, product in enumerate(products, 1):
    pt = my_with_wm.filter(pl.col("product") == product).sort("timestamp")
    if pt.height == 0:
        continue

    # 颜色：正 edge 绿，零 edge 灰，负 edge 红
    colors = ["green" if e > 0 else ("gray" if e == 0 else "red") for e in pt["edge"].to_list()]

    fig.add_trace(go.Scatter(
        x=pt["timestamp"].to_list(), y=pt["edge"].to_list(),
        mode="markers",
        marker=dict(color=colors, size=8, line=dict(width=0.5, color="black")),
        customdata=list(zip(pt["quantity"].to_list(), pt["trade_type"].to_list(), pt["price"].to_list())),
        hovertemplate="t=%{x}<br>edge=%{y}<br>qty=%{customdata[0]}<br>%{customdata[1]} @ %{customdata[2]}<extra></extra>",
        name=product, showlegend=False,
    ), row=i, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=0.5, row=i, col=1)
    fig.update_yaxes(title_text="Edge", row=i, col=1)

fig.update_xaxes(title_text="Timestamp", row=len(products), col=1)
fig.update_layout(height=350 * len(products), width=1200)
fig.show()

### 5c — 成交档位分布（normalized level histogram）

我方成交落在哪些 `price - fair` 档位，和预期策略设计对比。

In [42]:
fig = make_subplots(rows=1, cols=len(products),
                    subplot_titles=[f"{p} — Fill Level Distribution" for p in products])

for i, product in enumerate(products, 1):
    pt = my_with_wm.filter(pl.col("product") == product)
    if pt.height == 0:
        continue

    # norm_level = price - fair; 买入为负 edge，卖出为正 edge
    norm = (pt["price"] - pt["fair"]).to_list()

    # 分 buy/sell 颜色
    buys_norm = pt.filter(pl.col("trade_type") == "my_buy")
    sells_norm = pt.filter(pl.col("trade_type") == "my_sell")

    if buys_norm.height > 0:
        fig.add_trace(go.Histogram(
            x=(buys_norm["price"] - buys_norm["fair"]).to_list(),
            marker_color="limegreen", opacity=0.7, name="Buy", legendgroup="buy",
            showlegend=(i == 1),
        ), row=1, col=i)
    if sells_norm.height > 0:
        fig.add_trace(go.Histogram(
            x=(sells_norm["price"] - sells_norm["fair"]).to_list(),
            marker_color="red", opacity=0.7, name="Sell", legendgroup="sell",
            showlegend=(i == 1),
        ), row=1, col=i)

    fig.add_vline(x=0, line_dash="dash", line_color="black", line_width=1, row=1, col=i)
    fig.update_xaxes(title_text="Price - Fair", row=1, col=i)

fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_layout(height=400, width=1200, barmode="overlay",
                  legend=dict(orientation="h", y=1.08, x=0))
fig.show()

## Step 6: 成交间隔 & 活跃度分析

观察我方成交的时间分布：是否均匀地参与了全天成交机会？空闲时段是否是因为没有机会还是策略遗漏？

In [43]:
# 成交间隔直方图（每产品）
fig = make_subplots(rows=1, cols=len(products),
                    subplot_titles=[f"{p} — Trade Interval (ms)" for p in products])

for i, product in enumerate(products, 1):
    pt = my_trades.filter(pl.col("product") == product).sort("timestamp")
    if pt.height < 2:
        continue
    intervals = pt["timestamp"].diff().drop_nulls().to_list()

    fig.add_trace(go.Histogram(
        x=intervals, nbinsx=40, marker_color="steelblue",
        name=product, showlegend=False,
    ), row=1, col=i)
    fig.update_xaxes(title_text="Interval (ms)", row=1, col=i)

    mean_iv = np.mean(intervals)
    median_iv = np.median(intervals)
    print(f"{product}: {len(intervals)} intervals, mean={mean_iv:.0f}ms, median={median_iv:.0f}ms, "
          f"max={max(intervals)}ms, max_gap_at_ts={pt['timestamp'].to_list()[np.argmax(intervals)+1]}")

fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_layout(height=350, width=1200)
fig.show()

INTARIAN_PEPPER_ROOT: 23 intervals, mean=4017ms, median=2600ms, max=23900ms, max_gap_at_ts=53200


## Step 7: PnL 归因 — 做市 vs 头寸损益

将 PnL 拆分为：
- **Realized edge PnL**：每笔成交时的 edge × qty 累计（确定性收益）
- **Mark-to-market PnL**：持仓的未实现盈亏（随价格漂移波动）

两者之和应近似等于总 PnL。

In [44]:
fig = make_subplots(rows=len(products), cols=1, shared_xaxes=True,
                    subplot_titles=[f"{p} — PnL Attribution" for p in products],
                    vertical_spacing=0.08)

for i, product in enumerate(products, 1):
    pt = my_with_wm.filter(pl.col("product") == product).sort("timestamp")
    if pt.height == 0:
        continue

    # 累计 edge PnL
    ts_list = pt["timestamp"].to_list()
    cum_edge = np.cumsum(pt["edge_pnl"].to_list())

    # 总 PnL（从 activitiesLog）
    pnl_p = pnl_by_product.filter(pl.col("product") == product).sort("timestamp")

    # Mark-to-market 差值 = total_pnl - cum_edge（在成交时刻内插）
    fig.add_trace(go.Scatter(
        x=ts_list, y=cum_edge.tolist(),
        mode="lines+markers", line=dict(color="green", width=1.5),
        marker=dict(size=4),
        name="Cum Edge PnL", legendgroup="edge", showlegend=(i == 1),
    ), row=i, col=1)

    fig.add_trace(go.Scatter(
        x=pnl_p["timestamp"].to_list(), y=pnl_p["profit_and_loss"].to_list(),
        mode="lines", line=dict(color="orange", width=1.5),
        name="Total PnL", legendgroup="total", showlegend=(i == 1),
    ), row=i, col=1)

    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=0.5, row=i, col=1)
    fig.update_yaxes(title_text="PnL", row=i, col=1)

    final_edge = cum_edge[-1] if len(cum_edge) else 0
    final_total = pnl_p["profit_and_loss"].to_list()[-1] if pnl_p.height else 0
    print(f"{product}: cum_edge={final_edge:.2f}, total_pnl={final_total:.2f}, "
          f"mtm_component={final_total - final_edge:.2f}")

fig.update_xaxes(title_text="Timestamp", row=len(products), col=1)
fig.update_layout(height=400 * len(products), width=1200,
                  legend=dict(orientation="h", y=1.03, x=0))
fig.show()

INTARIAN_PEPPER_ROOT: cum_edge=-312.00, total_pnl=7563.00, mtm_component=7875.00


## Step 8: 汇总统计表

一目了然的关键指标。

In [45]:
summary_rows = []
for product in products:
    pt = my_with_wm.filter(pl.col("product") == product)
    pos_p = position_df.filter(pl.col("product") == product)
    pnl_p = pnl_by_product.filter(pl.col("product") == product).sort("timestamp")

    n_trades = pt.height
    total_vol = pt["quantity"].sum() if n_trades > 0 else 0
    n_pos_edge = pt.filter(pl.col("edge") > 0).height
    n_zero_edge = pt.filter(pl.col("edge") == 0).height
    n_neg_edge = pt.filter(pl.col("edge") < 0).height
    avg_edge = pt["edge"].mean() if n_trades > 0 else 0
    cum_edge_pnl = pt["edge_pnl"].sum() if n_trades > 0 else 0
    final_pnl = pnl_p["profit_and_loss"].to_list()[-1] if pnl_p.height else 0
    max_pos = pos_p["position"].max()
    min_pos = pos_p["position"].min()
    final_pos = pos_p["position"].to_list()[-1]

    # 成交频率（trades per 1000 ts）
    if n_trades >= 2:
        ts_range = pt["timestamp"].max() - pt["timestamp"].min()
        freq = n_trades / (ts_range / 1000) if ts_range > 0 else 0
    else:
        freq = 0

    summary_rows.append({
        "product": product,
        "trades": n_trades,
        "total_vol": total_vol,
        "pos_edge": n_pos_edge,
        "zero_edge": n_zero_edge,
        "neg_edge": n_neg_edge,
        "avg_edge": round(avg_edge, 2),
        "cum_edge_pnl": round(cum_edge_pnl, 2),
        "final_pnl": round(final_pnl, 2),
        "mtm_component": round(final_pnl - cum_edge_pnl, 2),
        "max_pos": max_pos,
        "min_pos": min_pos,
        "final_pos": final_pos,
        "trades_per_1k_ts": round(freq, 2),
    })

summary_df = pl.DataFrame(summary_rows)
print(summary_df)

shape: (2, 14)
┌───────────────┬────────┬───────────┬──────────┬───┬─────────┬─────────┬───────────┬──────────────┐
│ product       ┆ trades ┆ total_vol ┆ pos_edge ┆ … ┆ max_pos ┆ min_pos ┆ final_pos ┆ trades_per_1 │
│ ---           ┆ ---    ┆ ---       ┆ ---      ┆   ┆ ---     ┆ ---     ┆ ---       ┆ k_ts         │
│ str           ┆ i64    ┆ i64       ┆ i64      ┆   ┆ i64     ┆ i64     ┆ i64       ┆ ---          │
│               ┆        ┆           ┆          ┆   ┆         ┆         ┆           ┆ f64          │
╞═══════════════╪════════╪═══════════╪══════════╪═══╪═════════╪═════════╪═══════════╪══════════════╡
│ ASH_COATED_OS ┆ 0      ┆ 0         ┆ 0        ┆ … ┆ 0       ┆ 0       ┆ 0         ┆ 0.0          │
│ MIUM          ┆        ┆           ┆          ┆   ┆         ┆         ┆           ┆              │
│ INTARIAN_PEPP ┆ 24     ┆ 122       ┆ 16       ┆ … ┆ 80      ┆ 0       ┆ 80        ┆ 0.26         │
│ ER_ROOT       ┆        ┆           ┆          ┆   ┆         ┆         ┆   